# 07 — Dashboard (exercício)

**Camada:** consumo (Gold → BI/IA).

**Objetivo:** transformar marts e views em **11 visualizações estáticas** + um `dashboard.html` consolidado, e validar o app interativo (`make dashboard`).

## Como o notebook está organizado

Mesmo plano da solução: cada chart vive em **sua própria seção** (markdown explicativo + uma célula de código).

| Seção | Pergunta de negócio | Fonte | Status |
| --- | --- | --- | --- |
| 1 | KPIs gerais (p50/p90, volume) | `mart_team_sla_and_outcomes` | **TODO** |
| 2 | Onde o risco se concentra? | `mart_asset_risk_by_type` | **TODO** |
| 3 | Mix de severidade 100% | `mart_asset_risk_by_type` | já preenchido |
| 4 | Volume × velocidade por time | `mart_team_sla_and_outcomes` | já preenchido |
| 5 | Tendência mensal | `vw_calendarized_asset_type_trend` | **TODO** |
| 6 | Bounty acelera disclosure? | `mart_bounty_impact` | já preenchido |
| 7 | Heatmap Team × Asset | `vw_program_health_snapshot` | já preenchido |
| 8 | Tendência trimestral de weaknesses | `mart_weakness_trend_qtr` | já preenchido |
| 9 | Qualidade dos pesquisadores | `mart_reporter_quality` | já preenchido |
| 10 | Bounty × duplicatas | `mart_bounty_impact` | já preenchido |
| 11 | Bounty × informativos | `mart_bounty_impact` | já preenchido |

> Você precisa preencher **3 seções** (1, 2 e 5). As demais já vêm completas para servir de referência. Use o gabarito (`07_dashboard_solucao.ipynb`) só como último recurso.

## Critérios de aceite

1. Os 3 TODOs rodam e geram um PNG em `data/charts/`.
2. A célula final escreve `data/charts/dashboard.html` referenciando as 11 imagens.
3. O `assert len(saved_paths) == 11` passa.


## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (consumo do **gold**):

> Fonte → Bronze → Staging → Dim/Fact → Checks → Profiling → Marts/Views → **Dashboard**

Este notebook não toca mais em `fact_report` — ele consome os marts e views que foram construídos sobre o star schema acima. As 11 figuras + `dashboard.html` são a "interface" final pra quem não escreve SQL.

## 0. Setup

Mesma célula de parâmetros e helpers usados na solução. Não precisa mexer.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB:    {DB_PATH}")
print(f"Charts em:       {CHARTS_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")


### Helpers

`qdf` executa SQL no DuckDB; `save_fig` persiste o PNG em `CHARTS_DIR` e registra o caminho em `saved_paths`; `safe_xticks` trunca rótulos longos.

In [ ]:
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
saved_paths: list[str] = []


def qdf(sql: str) -> pd.DataFrame:
    try:
        return con.sql(sql).df()
    except Exception as exc:
        print(f"Query falhou: {exc}\nSQL:\n{sql}")
        return pd.DataFrame()


def save_fig(fig, filename: str) -> str:
    path = CHARTS_DIR / f"{filename}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_paths.append(str(path))
    print(f"  → {path.name}")
    return str(path)


def safe_xticks(names) -> list[str]:
    out = [str(x) if x is not None else "" for x in names]
    return [n if len(n) <= 40 else n[:37] + "..." for n in out]


## 1. KPIs gerais — p50, p90 e volume total  *(TODO)*

**Pergunta:** quanto tempo um report típico leva até ser divulgado (p50)? E na cauda lenta (p90)? Quantos reports temos no total?

**Tarefas:**
1. Escreva uma query em `mart_team_sla_and_outcomes` calculando uma **média ponderada** de `p50_days_to_disclosure` e `p90_days_to_disclosure` (peso = `reports_total`) e o `SUM(reports_total)`.
2. Renderize esses 3 valores como texto numa figura `2.4` × `8` polegadas (sem eixos) e salve com `save_fig(fig, "01_kpis_p50_p90_reports")`.


In [ ]:
df_kpi = qdf("""
SELECT
    SUM(p50_days_to_disclosure * reports_total) / NULLIF(SUM(reports_total), 0) AS kpi_p50_days,
    SUM(p90_days_to_disclosure * reports_total) / NULLIF(SUM(reports_total), 0) AS kpi_p90_days,
    SUM(reports_total) AS reports_total
FROM mart_team_sla_and_outcomes;
""")

p50 = float(df_kpi["kpi_p50_days"].fillna(0).iloc[0]) if not df_kpi.empty else 0.0
p90 = float(df_kpi["kpi_p90_days"].fillna(0).iloc[0]) if not df_kpi.empty else 0.0
volume = int(df_kpi["reports_total"].fillna(0).iloc[0]) if not df_kpi.empty else 0

fig = plt.figure(figsize=(8, 2.4))
plt.axis("off")
fig.text(0.02, 0.78, "KPIs gerais", fontsize=15, fontweight="bold")
fig.text(0.02, 0.38, f"p50 disclosure\n{p50:,.1f} dias", fontsize=12)
fig.text(0.36, 0.38, f"p90 disclosure\n{p90:,.1f} dias", fontsize=12)
fig.text(0.70, 0.38, f"Reports\n{volume:,}", fontsize=12)
save_fig(fig, "01_kpis_p50_p90_reports")


## 2. Onde o risco se concentra? *(TODO)*

**Pergunta:** quais tipos de ativo recebem mais reports?

**Tarefas:**
1. Selecione `asset_type, total_reports` de `mart_asset_risk_by_type` ordenado por volume.
2. **Filtre `asset_type IS NOT NULL`** — reports sem asset_type estruturado distorcem o ranking e quebram o matplotlib (Arrow `StringDtype` mantém NaN como float).
3. Plote um bar chart (`asset_type` no X, `total_reports` no Y), girando os rótulos do eixo X em 45° com `safe_xticks(...)`.
4. Salve como `02_asset_concentration_bar`.


In [ ]:
asset = qdf("""
SELECT asset_type, total_reports
FROM mart_asset_risk_by_type
WHERE asset_type IS NOT NULL
ORDER BY total_reports DESC;
""")

fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(len(asset))
ax.bar(x, asset["total_reports"])
ax.set_title("Reports por tipo de ativo")
ax.set_ylabel("Total de reports")
ax.set_xticks(x)
ax.set_xticklabels(safe_xticks(list(asset["asset_type"])), rotation=45, ha="right")
fig.tight_layout()
save_fig(fig, "02_asset_concentration_bar")


## 3. Mix de severidade por tipo de ativo *(referência)*

Stacked bar **100%** já implementado: facilita comparar o mix mesmo entre asset types com volumes muito diferentes.

In [ ]:
sev = qdf("""
SELECT asset_type, critical_cnt, high_cnt, medium_cnt, low_cnt
FROM mart_asset_risk_by_type
""").dropna(subset=["asset_type"])

cols  = ["critical_cnt", "high_cnt", "medium_cnt", "low_cnt"]
total = sev[cols].sum(axis=1).replace(0, np.nan)
for col in cols:
    sev[col] = sev[col] / total

fig, ax = plt.subplots(figsize=(7, 3))
bottom = np.zeros(len(sev))
for col in cols:
    valores = sev[col].fillna(0).to_numpy()
    ax.bar(sev["asset_type"].astype(str), valores, bottom=bottom, label=col.replace("_cnt", ""))
    bottom += valores

ax.set_title("Mix de severidade por asset type (100%)")
ax.set_ylim(0, 1)
ax.set_ylabel("Proporção")
ax.set_xticklabels(safe_xticks(list(sev["asset_type"])), rotation=45, ha="right")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
save_fig(fig, "03_severity_mix_100pct")


## 4. Volume × velocidade por time *(referência)*

Scatter `reports_total × p50_days_to_disclosure`, anotando os 5 times mais movimentados.

In [ ]:
team = qdf("""
SELECT team_handle, reports_total, p50_days_to_disclosure
FROM mart_team_sla_and_outcomes
WHERE team_handle IS NOT NULL
""")

fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(team["reports_total"], team["p50_days_to_disclosure"])
for _, row in team.sort_values("reports_total", ascending=False).head(5).iterrows():
    ax.annotate(str(row["team_handle"]),
                (row["reports_total"], row["p50_days_to_disclosure"]),
                fontsize=8)
ax.set_title("Volume vs velocidade por time")
ax.set_xlabel("Reports total")
ax.set_ylabel("p50 (dias até disclosure)")
fig.tight_layout()
save_fig(fig, "04_team_volume_vs_p50_scatter")


## 5. Tendência mensal por tipo de ativo *(TODO)*

**Pergunta:** o volume está subindo, caindo ou é sazonal?

**Tarefas:**
1. Consulte `vw_calendarized_asset_type_trend` (`month, asset_type, reports`).
2. Converta `month` para `datetime` com `pd.to_datetime`.
3. Identifique os **5** maiores `asset_type` por soma de `reports`.
4. Faça um line chart com 5 séries (uma por asset_type).
5. Salve como `05_monthly_trend_asset_type`.


In [ ]:
month = qdf("""
SELECT month, asset_type, reports
FROM vw_calendarized_asset_type_trend
""").dropna(subset=["month", "asset_type", "reports"])

month["month"] = pd.to_datetime(month["month"])
top_types: list[str] = (
    month.groupby("asset_type")["reports"].sum()
    .sort_values(ascending=False)
    .head(5)
    .index.astype(str)
    .tolist()
)

fig, ax = plt.subplots(figsize=(7, 3))
for asset_type in top_types:
    sub = month[month["asset_type"].astype(str) == asset_type].sort_values("month")
    ax.plot(sub["month"], sub["reports"], label=str(asset_type))
ax.set_title("Tendencia mensal - top 5 asset types")
ax.set_xlabel("Mes")
ax.set_ylabel("Reports")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "05_monthly_trend_asset_type")


## 6. Bounty acelera disclosure? *(referência)*

Grouped bar `p50` para `has_bounty=TRUE` vs `FALSE` por asset type.

In [ ]:
bounty = qdf("""
SELECT asset_type, has_bounty, p50_days_to_disclosure
FROM mart_bounty_impact
""")

piv = (bounty.pivot_table(index="asset_type", columns="has_bounty",
                          values="p50_days_to_disclosure", aggfunc="mean")
              .dropna(how="all"))

x = np.arange(len(piv))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 3))
if True in piv.columns:
    ax.bar(x - w / 2, piv[True].values,  w, label="Com bounty")
if False in piv.columns:
    ax.bar(x + w / 2, piv[False].values, w, label="Sem bounty")
ax.set_title("Bounty acelera disclosure? (p50 dias)")
ax.set_ylabel("p50 (dias)")
ax.set_xticks(x)
ax.set_xticklabels(safe_xticks(list(piv.index)), rotation=45, ha="right")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "06_bounty_p50_grouped")


## 7. Snapshot Team × Asset Type *(referência)*

Heatmap (linhas = times, colunas = asset types, intensidade = reports).

In [ ]:
snap = qdf("""
SELECT team_handle, asset_type, reports
FROM vw_program_health_snapshot
""").dropna()

mat = snap.pivot_table(index="team_handle", columns="asset_type",
                       values="reports", aggfunc="sum").fillna(0)

fig, ax = plt.subplots(figsize=(7, 3.6))
im = ax.imshow(mat.values, aspect="auto")
fig.colorbar(im, ax=ax, label="Reports")
ax.set_title("Heatmap Team × Asset Type")
ax.set_yticks(np.arange(len(mat.index)))
ax.set_yticklabels(mat.index.astype(str), fontsize=7)
ax.set_xticks(np.arange(len(mat.columns)))
ax.set_xticklabels(safe_xticks(list(mat.columns)), rotation=45, ha="right", fontsize=8)
fig.tight_layout()
save_fig(fig, "07_heatmap_team_asset")


## 8. Tendência trimestral das principais weaknesses *(referência)*

Line chart das 5 weaknesses mais reportadas no período total.

In [ ]:
weak = qdf("""
SELECT quarter, weakness_name, reports
FROM mart_weakness_trend_qtr
""").dropna()
weak["quarter"] = pd.to_datetime(weak["quarter"])

top_weak = (
    weak.groupby("weakness_name")["reports"].sum()
    .sort_values(ascending=False).head(5).index.tolist()
)

fig, ax = plt.subplots(figsize=(7, 3))
for nome in top_weak:
    sub = weak[weak["weakness_name"] == nome].sort_values("quarter")
    ax.plot(sub["quarter"], sub["reports"], label=str(nome))
ax.set_title("Top 5 weaknesses ao longo do tempo")
ax.set_xlabel("Trimestre")
ax.set_ylabel("Reports")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "08_weakness_trend_quarterly")


## 9. Produtividade × bounty hit-rate *(referência)*

Scatter `reports_total × bounty_rate_pct` para pesquisadores, anotando os 5 mais produtivos.

In [ ]:
rq = qdf("""
SELECT username, reports_total, bounty_rate_pct
FROM mart_reporter_quality
WHERE username IS NOT NULL
""")

fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(rq["reports_total"], rq["bounty_rate_pct"])
for _, row in rq.sort_values("reports_total", ascending=False).head(5).iterrows():
    ax.annotate(str(row["username"]),
                (row["reports_total"], row["bounty_rate_pct"]),
                fontsize=8)
ax.set_title("Produtividade vs bounty hit-rate")
ax.set_xlabel("Reports total")
ax.set_ylabel("Bounty rate (%)")
fig.tight_layout()
save_fig(fig, "09_reporter_quality_scatter")


## 10. Bounty reduz duplicatas? *(referência)*

In [ ]:
dup = qdf("""
SELECT asset_type, has_bounty, duplicate_cnt
FROM mart_bounty_impact
""").dropna(subset=["asset_type"])

piv = (dup.pivot_table(index="asset_type", columns="has_bounty",
                       values="duplicate_cnt", aggfunc="sum")
           .dropna(how="all"))

x = np.arange(len(piv))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 3))
if True in piv.columns:
    ax.bar(x - w / 2, piv[True].values,  w, label="Com bounty")
if False in piv.columns:
    ax.bar(x + w / 2, piv[False].values, w, label="Sem bounty")
ax.set_title("Duplicatas — com vs sem bounty")
ax.set_ylabel("Duplicate count")
ax.set_xticks(x)
ax.set_xticklabels(safe_xticks(list(piv.index)), rotation=45, ha="right")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "10_bounty_duplicates_grouped")


## 11. Bounty reduz informativos? *(referência)*

In [ ]:
inf = qdf("""
SELECT asset_type, has_bounty, informative_cnt
FROM mart_bounty_impact
""").dropna(subset=["asset_type"])

piv = (inf.pivot_table(index="asset_type", columns="has_bounty",
                       values="informative_cnt", aggfunc="sum")
           .dropna(how="all"))

x = np.arange(len(piv))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 3))
if True in piv.columns:
    ax.bar(x - w / 2, piv[True].values,  w, label="Com bounty")
if False in piv.columns:
    ax.bar(x + w / 2, piv[False].values, w, label="Sem bounty")
ax.set_title("Informativos — com vs sem bounty")
ax.set_ylabel("Informative count")
ax.set_xticks(x)
ax.set_xticklabels(safe_xticks(list(piv.index)), rotation=45, ha="right")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "11_bounty_informative_grouped")


## 12. Entregável estático: `dashboard.html`

Concatena as 11 figuras em um HTML simples — pronto para anexar em PR, e-mail ou Confluence.

Se algum dos seus TODOs anteriores não chamou `save_fig(...)`, o `assert` final vai falhar — perfeito para validar o pipeline.

In [ ]:
html_path = CHARTS_DIR / "dashboard.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Sessão 02 — Dashboard</title>")
    f.write("<style>body{font-family:sans-serif;max-width:900px;margin:24px auto;padding:0 16px}"
            "h1{border-bottom:1px solid #ddd;padding-bottom:8px}"
            "img{max-width:100%;border:1px solid #ddd;padding:4px;margin:16px 0}</style>")
    f.write("</head><body>")
    f.write("<h1>Sessão 02 — Dashboard HackerOne</h1>")
    for path in saved_paths:
        f.write(f"<div><img src='{Path(path).name}'></div>")
    f.write("</body></html>")

print(f"Dashboard HTML: {html_path}")
print(f"Figuras geradas: {len(saved_paths)}")
assert len(saved_paths) == 11, f"Esperava 11 figuras, encontrei {len(saved_paths)} — verifique seus TODOs."


## 13. Próximo passo: dashboard interativo (Streamlit + Plotly)

Os PNGs são bons como **snapshot**, mas para exploração ad-hoc usamos o app Streamlit em `dashboard/app.py`:

```bash
make dashboard      # http://localhost:8501
```

### Desafio extra

Depois de completar os 3 TODOs, abra o app e:

- filtre tipos de ativo na sidebar;
- altere o Top N;
- compare as abas de risco, SLA, bounty e IA;
- proponha **uma recomendação de negócio** baseada nos gráficos (ex.: "investir em bug bounty para asset_type X reduz p50 em Y dias").

A célula abaixo apenas valida que o app está presente.

In [ ]:
streamlit_app = PROJECT_ROOT / "dashboard" / "app.py"
assert streamlit_app.exists(), f"App Streamlit não encontrado: {streamlit_app}"
print("Dashboard interativo:", streamlit_app)
print("Para abrir:           make dashboard")
